In [50]:
import numpy as np
from numpy.linalg import inv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.datasets import make_spd_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

%matplotlib inline

In [51]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, rtol=1e-05, atol=1e-08):
    return np.allclose(a, a.T, rtol=rtol, atol=atol)

In [52]:
dim = 4
M = 2

In [53]:
G_s = []
sigmas = np.random.rand(M)
covar = np.zeros((dim, dim))
for i in range(M):
    mat = make_spd_matrix(dim)
    G_s.append(mat)
    covar += sigmas[i]*mat
G_s = np.array(G_s)
covar

array([[ 1.36130524,  0.60150484,  0.69038067, -0.19145898],
       [ 0.60150484,  1.72136192, -0.09137377, -0.26492294],
       [ 0.69038067, -0.09137377,  1.86571774, -0.06596241],
       [-0.19145898, -0.26492294, -0.06596241,  0.69461577]])

In [54]:
is_pos_def(G_s[0]), is_symmetric(G_s[0]), is_pos_def(G_s[1]), is_symmetric(G_s[1])

(True, True, True, True)

In [55]:
sigmas

array([0.38577112, 0.4222498 ])

In [56]:
covar

array([[ 1.36130524,  0.60150484,  0.69038067, -0.19145898],
       [ 0.60150484,  1.72136192, -0.09137377, -0.26492294],
       [ 0.69038067, -0.09137377,  1.86571774, -0.06596241],
       [-0.19145898, -0.26492294, -0.06596241,  0.69461577]])

In [57]:
covar.shape

(4, 4)

In [58]:
is_symmetric(covar), is_pos_def(covar)

(True, True)

In [59]:
N = 200
data_sim = np.random.multivariate_normal(np.zeros(dim), covar, N).T
C = np.cov(data_sim)
C, covar

(array([[ 1.28792518,  0.49438248,  0.62539603, -0.14777707],
        [ 0.49438248,  1.62096143, -0.07443934, -0.26073133],
        [ 0.62539603, -0.07443934,  1.80911629, -0.07734094],
        [-0.14777707, -0.26073133, -0.07734094,  0.75752405]]),
 array([[ 1.36130524,  0.60150484,  0.69038067, -0.19145898],
        [ 0.60150484,  1.72136192, -0.09137377, -0.26492294],
        [ 0.69038067, -0.09137377,  1.86571774, -0.06596241],
        [-0.19145898, -0.26492294, -0.06596241,  0.69461577]]))

In [60]:
def calc_matrices(sigma_hat, G_s, C, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim,))
    inv_sigma_hat = inv(sigma_hat) # calculate sigma_hat inverse
    sighat_c_sighat = (inv_sigma_hat.dot(C)).dot(inv_sigma_hat) # calculate sighat*C*sighat
    inv_calc = sighat_c_sighat-inv_sigma_hat # sighat_c_sighat - inv_sighat
    sub_calc = 2*(inv_sigma_hat.dot(C)).dot(inv_sigma_hat)-inv_sigma_hat # 2*inv_calc - sighat
    for h in range(dim):
        for g in range(dim):
            mult = ((inv_sigma_hat.dot(G_s[g])).dot(sub_calc)).dot(G_s[h])
            lhs = np.trace(mult)
            A[g, h] = lhs # because trace, the off diagonals are equal at i,j => j,i ?
        rhs = np.trace(inv_calc.dot(G_s[h]))
        B[h] = rhs
        
    return A, B

def iterative_soln(sig_zero, G_s, C, dim, iters=5):
    sig_imo = sig_zero
    for it in range(iters):
        sigma_hat = (sig_imo.reshape(-1, 1, 1)*G_s).sum(0) # calculate sigma_hat
        A, B = calc_matrices(sigma_hat, G_s, C, dim)
        r_i = inv(A).dot(B)
        sig_i = sig_imo + r_i
        print(sig_i)
        sig_imo = sig_i
    
    return sig_i

def unbiased_init(C, G_s, N, dim):
    A = np.zeros((dim, dim))
    B = np.zeros((dim, ))
    for g in range(dim):
        for h in range(dim):
            A[g, h] = np.trace(G_s[g].dot(G_s[h]))
        B[g] = (N/(N-1))*np.trace(C.dot(G_s[g]))
    return inv(A).dot(B)

def calc_likelihood(sigma, G_s, C, N, P):
    cov_hat = (sigma.reshape(-1, 1, 1)*G_s).sum(0)
    log_l = -P*np.log(2*np.pi) - np.log(np.linalg.det(cov_hat)) - np.trace(np.dot(inv(cov_hat), C))
    
    return log_l*(N/2)

def calc_likelihood_torch(sigma, G_s, C, N, P):
    C_t = torch.tensor(C)
    cov_hat = (sigma.reshape(-1, 1, 1)*G_s).sum(0)
    log_l = -P*np.log(2*np.pi) - torch.log(torch.linalg.det(cov_hat)) - torch.trace(
        torch.matmul(torch.linalg.inv(cov_hat), C_t))
    
    return log_l*(N/2)

In [61]:
sig_zero = unbiased_init(C=C, G_s=G_s, N=N, dim=M)
print(sig_zero, "\n")
sig_i = iterative_soln(sig_zero=sig_zero, G_s=G_s, C=C, dim=M, iters=20)

[0.35811702 0.40992584] 

[0.37769298 0.43596469]
[0.3800338  0.43969158]
[0.38006185 0.439756  ]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]
[0.38006186 0.43975602]


In [62]:
sig_i, sigmas

(array([0.38006186, 0.43975602]), array([0.38577112, 0.4222498 ]))

In [63]:
cov_hat = (sig_zero.reshape(-1, 1, 1)*G_s).sum(0)
cov_hat, covar

(array([[ 1.29845107,  0.55668429,  0.67333373, -0.17921796],
        [ 0.55668429,  1.61273988, -0.08449817, -0.24305869],
        [ 0.67333373, -0.08449817,  1.79640814, -0.06588403],
        [-0.17921796, -0.24305869, -0.06588403,  0.6571147 ]]),
 array([[ 1.36130524,  0.60150484,  0.69038067, -0.19145898],
        [ 0.60150484,  1.72136192, -0.09137377, -0.26492294],
        [ 0.69038067, -0.09137377,  1.86571774, -0.06596241],
        [-0.19145898, -0.26492294, -0.06596241,  0.69461577]]))

In [64]:
sum((sig_i-sigmas)**2), sum((sig_zero-sigmas)**2)

(0.00033906314736682097, 0.0009166292860176293)

In [65]:
calc_likelihood(sig_zero, G_s, C, N=N, P=dim)

-1200.6249775128013

In [66]:
calc_likelihood(sig_i, G_s, C, N=N, P=dim)

-1199.7531023845972

In [67]:
calc_likelihood_torch(torch.tensor(sig_i), torch.tensor(G_s), C, N=N, P=dim)

tensor(-1199.7531, dtype=torch.float64)

In [91]:
G_s_t = torch.nn.Parameter(torch.tensor(G_s))
calc_likelihood_torch(torch.tensor(sig_i), G_s_t, C, N=N, P=dim)

tensor(-1199.7531, dtype=torch.float64, grad_fn=<MulBackward0>)

In [94]:
optimizer = optim.Adam([G_s_t], lr=1e-3, betas=(0.9, 0.999))
for ep in range(100):
    optimizer.zero_grad()
    loss = -calc_likelihood_torch(torch.tensor(sig_i), G_s_t, C, N=N, P=dim)
    print(loss.item())
    loss.backward()
    optimizer.step()

1199.4956571932098
1199.4504742240433
1199.4074798284087
1199.3666715063507
1199.3280315521706
1199.2915234388806
1199.2570876769485
1199.2246378605373
1199.1940584126723
1199.1652061566506
1199.1379172831073
1199.1120189455632
1199.0873419684838
1199.063730654465
1199.0410479496084
1199.019176819879
1198.9980195870257
1198.9774964746762
1198.9575438875568
1198.9381125585655
1198.9191656087396
1198.900676582666
1198.8826275358545
1198.8650072384958
1198.8478095353096
1198.8310318786876
1198.814674038269
1198.7987369841235
1198.7832219398902
1198.768129603538
1198.7534595348222
1198.7392097088086
1198.7253762335197
1198.711953226665
1198.698932841745
1198.68630542794
1198.6740598018298
1198.6621836030374
1198.6506637013395
1198.6394866206163
1198.628638945822
1198.6181076830967
1198.607880549801
1198.5979461797297
1198.588294237914
1198.5789154480372
1198.5698015426551
1198.5609451515006
1198.552339646022
1198.543978959046
1198.535857397491
1198.5279694638127
1198.520309698784
1198.5128

In [95]:
G_s_t

Parameter containing:
tensor([[[ 1.3435,  1.5853, -0.2613, -0.3635],
         [ 1.5853,  3.4699, -0.2076, -0.8714],
         [-0.2613, -0.2076,  0.8111,  0.0967],
         [-0.3635, -0.8714,  0.0967,  1.1172]],

        [[ 1.8685, -0.1725,  1.7359, -0.0403],
         [-0.1725,  0.7322,  0.0672,  0.1507],
         [ 1.7359,  0.0672,  3.4956, -0.2751],
         [-0.0403,  0.1507, -0.2751,  0.7514]]], dtype=torch.float64,
       requires_grad=True)

In [96]:
G_s

array([[[ 1.4103558 ,  1.66301157, -0.1892549 , -0.40579989],
        [ 1.66301157,  3.5609808 , -0.25671097, -0.8619859 ],
        [-0.1892549 , -0.25671097,  0.90613756,  0.11264728],
        [-0.40579989, -0.8619859 ,  0.11264728,  1.05080232]],

       [[ 1.93541998, -0.09481825,  1.80791025, -0.08268352],
        [-0.09481825,  0.82330029,  0.01813595,  0.16010977],
        [ 1.80791025,  0.01813595,  3.59066133, -0.25913208],
        [-0.08268352,  0.16010977, -0.25913208,  0.68501295]]])